# `learn_averaged_templates_from_gpl` on LeiData

Pipeline on the first `N_WINDOWS` windows across LeiData SOCAL stems,
using [`learn_averaged_templates_from_gpl`](learn_dictionary.py):

1. Mean-center the spectrograms and width-filter GPL segments.
2. Call `learn_averaged_templates_from_gpl(..., n_windows, n_rounds)` which,
   each round on the current residual:
   - **Seed** (`Seed.from_gpl`)
   - **Average** matched patches
   - **Re-MP** with the averaged template
3. Inspect the returned `LearnedDictionary` (templates, matchings, residual).


In [ ]:
%matplotlib inline
%run seed_from_gpl_LeiData_setup.py


## Load windows + overlapping GPL detections


In [ ]:
# N_WINDOWS / N_OUTER_ITERS come from seed_from_gpl_LeiData_setup.py
# Load windows in stem order until we have N_WINDOWS (3×30 = 90 on this machine).

windows = []
segments = []
detections_total = 0

for gpl_dir in LEI_GPL_DIRS:
    if len(windows) >= N_WINDOWS:
        break
    npy_dir = gpl_dir / "spectrogram_npy"
    stem = gpl_dir.parent.name.replace(".gpl", "")
    csv_path = gpl_dir / f"CSV_{stem}.csv"
    if not csv_path.is_file():
        print(f"skip {stem}: missing {csv_path.name}")
        continue

    raw_npys = sorted(
        p
        for p in npy_dir.glob("*.npy")
        if not p.name.endswith("_display.npy")
    )
    need = N_WINDOWS - len(windows)
    take = raw_npys[:need]
    batch = [load_window(p) for p in take]

    for window in batch:
        mu = float(np.mean(window.data))
        window.data = mean_center(window.data)
        print(
            f"Mean-centered {stem} {window.window_id.split('_w')[-1][:6]} "
            f"(μ={mu:.6e}); new mean={float(np.mean(window.data)):.3e}"
        )

    detections = load_detections_csv(csv_path)
    detections_total += len(detections)
    overlapping = [
        d for d in detections if any(detection_overlaps_window(d, w) for w in batch)
    ]
    segs = detections_to_segments(batch, overlapping)
    segs = [s for s in segs if s.data.shape[1] < MAX_TEMPLATE_WIDTH]

    windows.extend(batch)
    segments.extend(segs)
    print(
        f"{stem}: loaded {len(batch)} windows, "
        f"overlapping={len(overlapping)}, width<{MAX_TEMPLATE_WIDTH} segments={len(segs)}"
    )

if len(windows) < N_WINDOWS:
    raise FileNotFoundError(
        f"Need {N_WINDOWS} windows, found only {len(windows)} under {LEI_DATA_ROOT}"
    )

print(f"\nTotal windows={len(windows)}, detections={detections_total}, segments={len(segments)}")
if segments:
    top = sorted(segments, key=lambda s: s.energy, reverse=True)[:10]
    print(f"Top-{len(top)} by energy (on mean-centered originals):")
    for s in top:
        print(
            f"  E={s.energy:.3e}  shape={s.data.shape}  "
            f"win={s.window_id.split('_w')[-1][:6]}  t=[{s.t_start},{s.t_end})"
        )
else:
    print(f"No segments with width < {MAX_TEMPLATE_WIDTH}")


## Learn averaged templates

`learn_averaged_templates_from_gpl` runs the outer residual-deflation loop and
returns a `LearnedDictionary` whose `get_seeds()` are the **refined**
(averaged-template) rounds — same `Seed` / `Template` / `Matching` format as
`seed_from_gpl`.


In [ ]:
if not segments:
    raise RuntimeError(f"No candidate segments with width < {MAX_TEMPLATE_WIDTH}")

import time

# Each outer round writes round_XXX/template.npy + matchings.json under TEMPLATE_OUT_DIR.
t0 = time.perf_counter()
result = learn_averaged_templates_from_gpl(
    windows,
    segments,
    n_windows=N_WINDOWS,
    n_rounds=N_OUTER_ITERS,
    eps_step_fraction=EPS_STEP_FRACTION,
    min_repeats=MIN_REPEATS,
    template_out_dir=TEMPLATE_OUT_DIR,
)
elapsed = time.perf_counter() - t0

seeds = result.get_seeds()
if not seeds:
    raise RuntimeError("learn_averaged_templates_from_gpl produced no rounds")

E0 = result.get_energy_before()
print(f"elapsed: {elapsed:.1f}s ({elapsed/60:.2f} min)")
cap = N_OUTER_ITERS if N_OUTER_ITERS is not None else "uncapped"
print(f"rounds completed: {len(seeds)} / {cap} (stop={result.get_stop_reason()})")
print(f"E0 (start) = {E0:.6e}")
print(f"E_final    = {result.get_energy_after():.6e} ({result.get_energy_after()/E0:.3%} of E0)")
print(f"ΔE_total   = {result.get_delta_energy_total():.6e}")
print(f"templates saved under: {TEMPLATE_OUT_DIR}")
print()
for i, seed in enumerate(seeds, start=1):
    print(
        f"  round {i}: matchings={len(seed.get_matchings())}  "
        f"ΔE={seed.get_delta_energy_total():.6e}  "
        f"E_after={seed.get_energy_after():.6e} "
        f"({seed.get_energy_after()/E0:.3%} of E0)"
    )
    tmpl = seed.get_template()
    if tmpl is not None:
        print(
            f"           template_id={tmpl.template_id}  "
            f"shape={tmpl.data.shape}  source={seed.get_selected_segment_id()}"
        )


## Plot helpers + energy trajectory


In [ ]:
def percentile_scale(arr, lo=5.0, hi=99.5):
    arr = np.asarray(arr)
    p_lo, p_hi = np.percentile(arr, [lo, hi])
    return p_lo, p_hi


def normalize_and_root(arr, lo, hi):
    """Scale to [0,1] for [lo,hi] interval, clip, then apply third root."""
    arr = np.clip((arr - lo) / (hi - lo), 0, 1)
    return np.cbrt(arr)


ref = windows[0]
f0, f1 = ref.start_freq, ref.end_freq
hop_sec = (ref.overlap / ref.sample_freq) if ref.sample_freq else 1.0
t_sec = ref.n_time * hop_sec

round_ids = list(range(1, len(seeds) + 1))
E_after = [s.get_energy_after() for s in seeds]
deltas = [s.get_delta_energy_total() for s in seeds]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(round_ids, [e / E0 for e in E_after], marker="o")
axes[0].set_xlabel("Outer round")
axes[0].set_ylabel("Residual energy / E0")
axes[0].set_title("Residual fraction after each round")
axes[0].set_xticks(round_ids[:: max(1, len(round_ids)//10)])
axes[0].grid(True, alpha=0.3)

axes[1].bar(round_ids, deltas, color="steelblue")
axes[1].set_xlabel("Outer round")
axes[1].set_ylabel("ΔE (refined)")
axes[1].set_title("Energy removed per round")
axes[1].set_xticks(round_ids[:: max(1, len(round_ids)//10)])
axes[1].grid(True, axis="y", alpha=0.3)

fig.tight_layout()
plt.show()


## Templates after each outer round (averaged)


In [ ]:
templates = result.get_templates()
n = len(templates)
nrows, ncols = TEMPLATE_GRID
if n > nrows * ncols:
    raise RuntimeError(f"Need a larger TEMPLATE_GRID: {n} templates > {nrows}×{ncols}")

fig, axes = plt.subplots(nrows, ncols, figsize=(2.4 * ncols, 2.6 * nrows))
axes_flat = np.atleast_1d(axes).ravel()

all_vals = np.concatenate([t.data.ravel() for t in templates])
lo, hi = percentile_scale(all_vals, lo=5, hi=99.5)

im = None
for idx, ax in enumerate(axes_flat):
    if idx >= n:
        ax.axis("off")
        continue
    tmpl = templates[idx]
    seed = seeds[idx]
    rid = round_ids[idx]
    w_bins = tmpl.width
    tmpl_sec = w_bins * hop_sec
    img = normalize_and_root(tmpl.data, lo, hi)
    im = ax.imshow(
        img,
        origin="lower",
        aspect="auto",
        cmap="jet",
        extent=[0, tmpl_sec, f0, f1],
        vmin=0,
        vmax=1,
    )
    ax.set_title(
        f"r{rid} m={len(seed.get_matchings())}\nΔE={seed.get_delta_energy_total():.2e}",
        fontsize=8,
    )
    ax.tick_params(labelsize=7)

if im is not None:
    fig.colorbar(
        im,
        ax=axes_flat[:n].tolist(),
        fraction=0.015,
        pad=0.01,
        label="shared [5–99.5%]+root",
    )
fig.suptitle(f"Averaged templates ({n} rounds)", y=1.01)
fig.tight_layout()
plt.show()


## Refined matchings (first `N_PLOT_ROUNDS` rounds × `N_PLOT_WINDOWS` windows)


In [ ]:
for i, seed in enumerate(seeds[:N_PLOT_ROUNDS]):
    template = seed.get_template()
    matchings = seed.get_matchings()
    if i == 0:
        residual_before = {w.window_id: w.data for w in windows}
    else:
        residual_before = seeds[i - 1].get_residual_after()

    w_bins = template.width if template is not None else 1
    tmpl_sec = w_bins * hop_sec
    if template is not None:
        tmpl_min, tmpl_max = percentile_scale(template.data, lo=5, hi=99.5)
        template_img = normalize_and_root(template.data, tmpl_min, tmpl_max)

    print(f"--- round {i + 1}: {len(matchings)} refined matchings ---")
    for window in windows[:N_PLOT_WINDOWS]:
        arr = residual_before[window.window_id]
        sp_min, sp_max = percentile_scale(arr, lo=5, hi=99.5)
        window_img = normalize_and_root(arr, sp_min, sp_max)
        win_matchings = [m for m in matchings if m.window_id == window.window_id]

        fig, axes = plt.subplots(
            1,
            2,
            figsize=(14, 4.5),
            gridspec_kw={"width_ratios": [t_sec, max(tmpl_sec, 1e-9)], "wspace": 0.35},
        )
        ax = axes[0]
        im = ax.imshow(
            window_img,
            origin="lower",
            aspect="auto",
            cmap="jet",
            extent=[0, t_sec, f0, f1],
            vmin=0,
            vmax=1,
        )
        for m in win_matchings:
            t0m = m.t_start * hop_sec
            t1m = (m.t_start + w_bins) * hop_sec
            ax.axvspan(t0m, t1m, color="black", alpha=0.08)
            ax.axvline(t0m, color="yellow", linewidth=2.0, alpha=0.95)
            ax.axvline(t1m, color="red", linewidth=2.0, alpha=0.95)
        ax.set_title(
            f"Round {i + 1} input residual "
            f"{window.window_id.split('_w')[-1][:6]} — {len(win_matchings)} matchings"
        )
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Frequency (Hz)")
        fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="rescaled [5–99.5%]+root")

        ax = axes[1]
        if template is not None:
            ax.imshow(
                template_img,
                origin="lower",
                aspect="auto",
                cmap="jet",
                extent=[0, tmpl_sec, f0, f1],
                vmin=0,
                vmax=1,
            )
            ax.axvline(0, color="yellow", linewidth=2.0, alpha=0.95)
            ax.axvline(tmpl_sec, color="red", linewidth=2.0, alpha=0.95)
            ax.set_title(
                f"Averaged template (round {i + 1})\n"
                f"{template.data.shape[0]}×{w_bins} bins ({tmpl_sec:.3f} s)"
            )
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Frequency (Hz)")
        else:
            ax.axis("off")

        fig.tight_layout()
        plt.show()


## Final residual vs original


In [ ]:
final_resid = result.get_residual_after()
if final_resid is None:
    raise RuntimeError("No final residual")

for window in windows[:N_PLOT_WINDOWS]:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    arrs = [window.data, final_resid[window.window_id]]
    titles = [
        f"Original mean-centered {window.window_id.split('_w')[-1][:6]}",
        f"Residual after {len(seeds)} outer rounds",
    ]
    for ax, arr, title in zip(axes, arrs, titles):
        lo, hi = percentile_scale(arr, lo=5, hi=99.5)
        ax.imshow(
            normalize_and_root(arr, lo, hi),
            origin="lower",
            aspect="auto",
            cmap="jet",
            extent=[0, t_sec, f0, f1],
            vmin=0,
            vmax=1,
        )
        ax.set_title(title)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Frequency (Hz)")
    fig.tight_layout()
    plt.show()

print(f"E0 = {E0:.6e}")
print(f"Final = {result.get_energy_after():.6e} ({result.get_energy_after()/E0:.3%} of E0)")
for i, seed in enumerate(seeds, start=1):
    print(
        f"  round {i}: removed {seed.get_delta_energy_total():.6e}  "
        f"remaining {seed.get_energy_after()/E0:.3%}"
    )
